# DBMS Phase 2 — The Formal Relational Model

**Roadmap source:** `dbms_complete_roadmap.md` → Phase 2: The Formal Relational Model (Lectures 2 and 3).

SQL practice is kept in the companion file [dbms_phase2_practice.sql](dbms_phase2_practice.sql). This notebook focuses on theory, reasoning, and small Python experiments that make the formal ideas concrete.

## Study method

For each topic: learn the definition → inspect the example → implement the `YOUR TURN` cell → explain the result in exam language.

# 1. Why the relational model exists

E. F. Codd proposed the relational model in 1970 to separate the logical organization of data from the details of machine storage. Instead of asking an application to follow pointers through files, the model represents data as relations and lets the DBMS evaluate declarative queries.

A relation is mathematically a subset of a Cartesian product of attribute domains:

$$R \subseteq D_1 \times D_2 \times \dots \times D_n$$

Interpretation:

- Each attribute has a domain of permitted values.
- A tuple chooses one value from each domain.
- The relation contains only the tuples that are currently true.
- A formal relation is a set: duplicate tuples do not exist, and tuple order has no meaning.

## Terms you must distinguish

| Term | Meaning | Example |
|---|---|---|
| Relation schema | Static structure/intension: relation name, attributes, domains, constraints | `Student(student_id: TEXT, name: TEXT)` |
| Relation instance | Current snapshot/extension: the tuples stored now | `{('S1', 'Mira'), ('S2', 'Arun')}` |
| Attribute | Named column of a relation | `student_id` |
| Domain | Set/type of values permitted for an attribute | non-negative integers |
| Tuple | One record/row in the relation | `('S1', 'Mira')` |
| Degree | Number of attributes/columns | A 2-column relation has degree 2 |
| Cardinality | Number of tuples/rows in an instance | Three stored tuples means cardinality 3 |

In [1]:
# A tiny formal-style relation instance. The schema is represented separately.
student_schema = ('student_id', 'name')
student_instance = {
    ('S1', 'Mira'),
    ('S2', 'Arun'),
}

print('schema:', student_schema)
print('degree:', len(student_schema))
print('cardinality:', len(student_instance))
print('instance:', student_instance)

schema: ('student_id', 'name')
degree: 2
cardinality: 2
instance: {('S2', 'Arun'), ('S1', 'Mira')}


## Atomic domains and First Normal Form

A relational attribute should draw one atomic value from its domain. A cell should not contain a repeating list of unrelated values when those values represent separate facts. This is the intuition behind First Normal Form (1NF).

Poor design: `Student(student_id, name, phone_numbers)` where one cell stores `['111', '222']`.

Relational design: keep `Student(student_id, name)` and create `StudentPhone(student_id, phone_number)` with one phone number per tuple. The companion SQL file demonstrates this style through enrollment and prerequisite relations.

# 2. Set semantics versus SQL bag semantics

Formal relational algebra treats a relation as a set: duplicate tuples are eliminated and there is no inherent order. SQL commonly uses **bag/multiset semantics**: a query can return duplicate rows unless `DISTINCT` is requested.

This difference matters in exams. Projection in formal relational algebra removes duplicate tuples; SQL `SELECT column` may retain duplicates, while `SELECT DISTINCT column` removes them.

In [ ]:
bag_of_departments = ['CS', 'CS', 'Math']
formal_set = set(bag_of_departments)

print('bag:', bag_of_departments)
print('set:', formal_set)
print('bag cardinality:', len(bag_of_departments))
print('set cardinality:', len(formal_set))

### Learn → implement: formal projection

A projection keeps selected attributes and removes duplicate resulting tuples under formal set semantics. Implement the Python analogue below.

Input records are dictionaries. Return a `set` of tuples in the requested column order. Do not use a database or SQL in this cell.

In [ ]:
# YOUR TURN
def project_as_relation(
    records: list[dict[str, str]],
    selected_attributes: tuple[str, ...],
) -> set[tuple[str, ...]]:
    raise NotImplementedError


In [ ]:
# Checks — run after implementing project_as_relation.
courses = [
    {'course_id': 'CS101', 'department': 'CS'},
    {'course_id': 'CS102', 'department': 'CS'},
    {'course_id': 'MA101', 'department': 'Math'},
]
assert project_as_relation(courses, ('department',)) == {('CS',), ('Math',)}
assert project_as_relation(courses, ('department', 'course_id')) == {
    ('CS', 'CS101'), ('CS', 'CS102'), ('Math', 'MA101')
}
print('Formal-projection checks passed.')

# 3. Key taxonomy

Keys are about uniqueness across every valid relation instance, not merely what happens to be unique in today's sample rows.

- **Superkey:** an attribute set that uniquely identifies every tuple. Extra attributes are allowed.
- **Candidate key:** a minimal superkey. Removing any attribute destroys the uniqueness guarantee.
- **Primary key:** the candidate key selected by the designer as the table's main identifier.
- **Composite key:** a key containing multiple attributes, such as `(student_id, course_id)`.
- **Foreign key:** attributes in one relation whose values reference a candidate/primary key in another relation.
- **Natural key:** meaningful real-world identifier, such as an institutional roll number.
- **Surrogate key:** generated identifier with no business meaning, such as an auto-increment integer.

Minimality is the exam distinction: every candidate key is a superkey, but a superkey need not be a candidate key. If `student_id` is unique, then `{student_id}` and `{student_id, name}` are superkeys; only `{student_id}` is minimal.

In [ ]:
def is_superkey(records: list[dict[str, str]], attributes: tuple[str, ...]) -> bool:
    """Return whether the selected attribute values are unique in this instance."""
    observed_values: set[tuple[str, ...]] = set()
    for record in records:
        key_value = tuple(record[attribute] for attribute in attributes)
        if key_value in observed_values:
            return False
        observed_values.add(key_value)
    return True

students = [
    {'student_id': 'S1', 'email': 'mira@example.edu', 'name': 'Mira'},
    {'student_id': 'S2', 'email': 'arun@example.edu', 'name': 'Arun'},
]
print(is_superkey(students, ('student_id',)))
print(is_superkey(students, ('student_id', 'name')))

### Learn → implement: candidate-key reasoning

Implement `is_candidate_key`. It should return `True` only if the attributes form a superkey **and** no proper single-attribute removal remains a superkey. For this exercise, checking removal of one attribute is sufficient because the inputs are small.

An empty attribute set is not a useful candidate key for this exercise.

In [ ]:
# YOUR TURN
def is_candidate_key(
    records: list[dict[str, str]],
    attributes: tuple[str, ...],
) -> bool:
    raise NotImplementedError


In [ ]:
# Checks — run after implementing is_candidate_key.
assert is_candidate_key(students, ('student_id',)) is True
assert is_candidate_key(students, ('email',)) is True
assert is_candidate_key(students, ('student_id', 'name')) is False
duplicate_names = [
    {'student_id': 'S1', 'name': 'Mira'},
    {'student_id': 'S2', 'name': 'Mira'},
]
assert is_candidate_key(duplicate_names, ('name',)) is False
print('Key-taxonomy checks passed.')

# 4. The three fundamental integrity constraints

### Domain integrity
Every value belongs to the attribute's domain: correct type, allowed range, format, and any `CHECK` predicate. Example: a course credit count must be a non-negative integer.

### Entity integrity
A primary-key value cannot be `NULL`. Every tuple needs an unambiguous identity, and no two tuples may share that identity.

### Referential integrity
A non-null foreign-key value must match an existing referenced primary/candidate key. This prevents an enrollment from referring to a student that does not exist. A foreign key may be entirely `NULL` when the relationship is optional.

The SQL companion demonstrates these with `PRIMARY KEY`, `NOT NULL`, `CHECK`, and `FOREIGN KEY`.

### Learn → implement: validate a tuple before insertion

Implement a small application-side validator to make the three ideas visible. This is not a replacement for database constraints; it is a preflight check that can produce a friendly error before the database rejects the write.

In [ ]:
# YOUR TURN
def validate_enrollment(
    enrollment: dict[str, str | int],
    known_student_ids: set[str],
    known_course_ids: set[str],
) -> None:
    # Validate student_id, course_id, semester, and year.
    # Raise ValueError with a useful message for the first invalid rule.
    raise NotImplementedError


In [ ]:
# Checks — run after implementing validate_enrollment.
known_students = {'S1', 'S2'}
known_courses = {'CS101'}
valid_enrollment = {'student_id': 'S1', 'course_id': 'CS101', 'semester': 'Fall', 'year': 2026}
assert validate_enrollment(valid_enrollment, known_students, known_courses) is None

invalid_enrollments = [
    {**valid_enrollment, 'student_id': 'S9'},
    {**valid_enrollment, 'course_id': 'CS999'},
    {**valid_enrollment, 'semester': ''},
    {**valid_enrollment, 'year': 1999},
]
for invalid_enrollment in invalid_enrollments:
    try:
        validate_enrollment(invalid_enrollment, known_students, known_courses)
    except ValueError:
        pass
    else:
        raise AssertionError(f'{invalid_enrollment} should be rejected')
print('Integrity-validation checks passed.')

# 5. Mapping relationships to tables

### One-to-many (1:N)
Place the primary key of the one-side relation as a foreign key in the many-side relation. Example: one department has many students, so `student.department_id` references `department.department_id`.

### Many-to-many (M:N)
Create an associative/intersection relation. Example: students take many courses and courses have many students, so `enrollment(student_id, course_id, semester, year)` references both sides. The participating foreign keys commonly form a composite primary key. Relationship attributes belong in the associative relation.

### Self-referencing relationship
A relation references its own primary key. Example: `prerequisite(course_id, prerequisite_id)` has two foreign keys, both referencing `course.course_id`; aliases are needed when querying the table twice.

The companion `.sql` file creates all three patterns.

In [ ]:
# Relationship mapping as metadata, before writing SQL.
relationship_recipes = {
    'one_to_many': 'department 1 ───< student; FK on student',
    'many_to_many': 'student >───< course; create enrollment(student_id, course_id)',
    'self_reference': 'course >─── prerequisite ───< course; two FKs to course',
}
for relationship, recipe in relationship_recipes.items():
    print(f'{relationship}: {recipe}')

## Exam traps to rehearse

- A superkey may contain unnecessary attributes; a candidate key may not.
- A primary key is a chosen candidate key, not every unique column.
- A foreign key references a key in another relation; it does not have to be the primary key if a candidate key is referenced and declared unique.
- Formal relations are unordered sets; SQL result order is not guaranteed without `ORDER BY`.
- SQL tables and query results can contain duplicate rows unless constrained or projected with `DISTINCT`.
- A foreign key prevents dangling references; it does not by itself make the relationship one-to-many or many-to-many. The schema shape does that.
- Application validation improves error messages, but database constraints are the authoritative protection against competing writers.

# Phase 2 exam checkpoint

Answer these without looking at the previous cells:

1. State the formal definition of a relation and explain domains, tuples, degree, and cardinality.
2. Distinguish relation schema from relation instance.
3. Why does formal projection remove duplicates while ordinary SQL projection may not?
4. Prove by reasoning why every candidate key is a superkey but not every superkey is a candidate key.
5. Contrast natural and surrogate keys with one trade-off each.
6. Give one example each of domain, entity, and referential integrity.
7. Map a student–course M:N relationship into tables and identify the composite key.
8. How is a self-referencing prerequisite relation represented?

Complete the `YOUR TURN` cells, run the companion [dbms_phase2_practice.sql](dbms_phase2_practice.sql), and send me one implementation or test failure at a time for review.